<a id="top"></a>

# Lab L0406: build a routing workflow (and swap the decider)

```yaml
title: "Lab L0406: build a routing workflow (and swap the decider)"
keywords: langgraph, routing, conditional edge, classifier, user-input branching, deterministic, eval, workflow vs agent, lab
estimated duration: 45
```

> **Lesson:** L04. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L04/objectives.md).
> **Solutions:** `L1106_lab_solutions.ipynb`.
> Runs **fully offline** -- a deterministic `StubChat` stands in for `ChatAnthropic`. No API key
> needed. Builds on [Lab L0404](L1104_lab_empty.ipynb).

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 -- Routing state + classify node](#2-problem-1----routing-state--classify-node)
- [3. Problem 2 -- The branch nodes](#3-problem-2----the-branch-nodes)
- [4. Problem 3 -- The conditional edge](#4-problem-3----the-conditional-edge)
- [5. Problem 4 -- Run and prove determinism](#5-problem-4----run-and-prove-determinism)
- [6. Problem 5 -- Same graph, user-input branch](#6-problem-5----same-graph-user-input-branch)
- [7. Problem 6 (optional) -- Evaluate the classifier](#7-problem-6-optional----evaluate-the-classifier)
- [8. Problem 7 -- Workflow vs. agent (written)](#8-problem-7----workflow-vs-agent-written)
- [9. Self-check](#9-self-check)

## 1. Setup

Given again: the offline `StubChat` (this one also returns a **category** when asked to classify),
the sample `TICKETS`, and the `haiku`/`sonnet` stub clients. Run this first. (Same setup as Lab
L0404 -- this lab stands alone.)

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, StateGraph

HAIKU = "claude-haiku-4-5"  # the names are real; here a stub stands in for the client
SONNET = "claude-sonnet-4-6"

TICKETS = {
    "billing": "I was charged twice for my subscription this month -- please refund the extra charge.",
    "technical": "The export button throws a 500 error every time I click it on the reports page.",
    "general": "Do you offer a student discount, and how would I apply it to my account?",
}
POLICY = (
    "Refunds: a duplicate charge is always refundable within 60 days. "
    "Never promise a refund for change-of-mind. "
    "Escalate anything mentioning legal action or chargebacks to a human."
)


class StubReply:
    """Mimics the one field we read off a ChatAnthropic response: `.content`."""

    def __init__(self, content: str) -> None:
        self.content = content


class StubChat:
    """A tiny OFFLINE stand-in for ChatAnthropic, so this lab is free, fast, and deterministic.

    It mimics the single call our nodes use -- `.invoke(prompt).content` -> str. The graph
    wiring you practice here is identical with a real model: swapping `StubChat(HAIKU)` for
    `ChatAnthropic(model=HAIKU, api_key=require_anthropic_key())` is a one-line change, and the
    node code never changes (Problem 5). The lecture demos use the real client.
    """

    def __init__(self, model: str) -> None:
        self.model = model

    def invoke(self, prompt: str) -> StubReply:
        p = prompt.lower()
        if "classify" in p:
            # Key on TICKET-content words only. We avoid "bill"/"technical"/"general" because
            # those leak from the instruction text itself ("billing, technical, or general").
            if any(w in p for w in ("charge", "charged", "refund", "invoice", "payment", "twice")):
                return StubReply("billing")
            if any(w in p for w in ("error", "500", "crash", "bug", "broken", "login", "log in")):
                return StubReply("technical")
            return StubReply("general")
        if "summarize" in p:
            return StubReply("The customer needs help resolving an account issue.")
        if "compliance" in p or "policy" in p:
            return StubReply("OK: the reply is consistent with the refund policy.")
        return StubReply("Thanks for reaching out -- happy to help you with this!")


# Per-node models -- a stub each, exactly where a real graph would bind Haiku vs. Sonnet.
haiku = StubChat(HAIKU)
sonnet = StubChat(SONNET)

[↑ Back to top](#top)

## 2. Problem 1 -- Routing state + classify node

Define `RouteState` (it needs `ticket`, `category`, `user_choice`, `draft`, and the `steps` append
list), then write the `classify` entry node. `classify` runs on the **cheap** stub (`haiku`) because
it only produces a one-word label; keep only `billing`/`technical`/`general` and default to
`general`.

In [ ]:
# TODO 1: complete RouteState -- ticket, category, user_choice, draft (all str) plus
#         steps: Annotated[list[str], add]  (the append reducer). REPLACE the 1-field stub below.
# TODO 2: write classify(state) -> dict:
#   call haiku.invoke("Classify ... billing/technical/general ...\n\n" + state["ticket"]),
#   keep only a known label (default "general"),
#   return {"category": <label>, "steps": [f"classify->{<label>}"]}
class RouteState(TypedDict):
    ticket: str


def classify(state: RouteState) -> dict[str, object]:
    raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 3. Problem 2 -- The branch nodes

Write the three specialized branch nodes -- `billing`, `technical`, `general`. Each runs on the
**capable** stub (`sonnet`), drafts a reply for `state["ticket"]`, and appends its name to `steps`.
The model works *inside* each node.

In [ ]:
# TODO: write three branch nodes -- billing(state), technical(state), general(state).
#   Each calls sonnet.invoke(...) with a branch-specific instruction + state["ticket"],
#   and returns {"draft": <text>, "steps": [<branch name>]}.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 4. Problem 3 -- The conditional edge

Write `route(state)` returning `state["category"]`, then wire it with `add_conditional_edges`. Say
the line as you go: **this is not the model deciding.** The model produced a *label*; your routing
function reads that label and picks the edge. (In L14 the routing function will branch on whether the
*model asked for a tool* -- same mechanism, different decider.) Compile to `route_app` and print the
diagram -- it now **branches**.

In [ ]:
# TODO 1: write route(state) -> str that returns state["category"].
#         (This is NOT the model deciding -- it reads a label already in state.)
# TODO 2: build the routing graph: add classify + the three branch nodes,
#         set_entry_point("classify"),
#         add_conditional_edges("classify", route, {"billing":"billing","technical":"technical","general":"general"}),
#         add_edge from each branch to END, compile() to `route_app`, and print draw_mermaid().
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 5. Problem 4 -- Run and prove determinism

Invoke once per category, then run the *same* ticket twice and show the path is identical. The
branch you land in is fixed by the classifier's label -- re-running never changes the path.

In [ ]:
# TODO: invoke route_app once per category in TICKETS and print f"{kind} -> {out['steps']}".
#       Then invoke the SAME ticket twice and show the path is identical (deterministic).
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 6. Problem 5 -- Same graph, user-input branch

Now swap the **decider**. Replace `classify` with an `intake` pass-through node and a
`route_by_user` function that reads `state["user_choice"]` -- a value the **user** supplied in the
initial state. **No model is involved in the routing decision at all.** Feed a *technical* ticket but
set `user_choice="billing"`: the path must follow the **user**, not the ticket -- *the user owns the
edge, the model still works inside the chosen branch node.*

In [ ]:
# TODO 1: write intake(state) -> dict: a plain pass-through node (no model), return {"steps": ["intake"]}.
# TODO 2: write route_by_user(state) -> str that returns state["user_choice"] (NO model in this decision).
# TODO 3: build user_app: intake entry -> conditional edge route_by_user to the three branches -> END.
# TODO 4: invoke with a TECHNICAL ticket but user_choice="billing"; print the path. It should follow
#         the USER's choice (billing), not the ticket content.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 7. Problem 6 (optional) -- Evaluate the classifier

Carry the **L12 eval discipline** onto your workflow -- it's the cheapest thing to test, because the
same input always takes the same path. Run the small `EVAL_CASES` set through your `classify` node
and report the pass rate. One case is ambiguous on purpose; let the eval *surface* the weakness
rather than hide it. (In L12 this lived in a reusable harness; here it's inlined.)

In [ ]:
# OPTIONAL. A deterministic workflow is the easiest thing to evaluate: same input -> same path.
# TODO: for each (ticket, expected) in EVAL_CASES, call classify({"ticket": ticket, "steps": []}),
#       compare its "category" to expected, print PASS/FAIL, and report the pass rate.
EVAL_CASES = [
    ("My card was charged twice for one order.", "billing"),
    ("The dashboard shows a 500 error on load.", "technical"),
    ("What are your support hours?", "general"),
    ("The app keeps crashing and I want a refund.", "technical"),  # ambiguous on purpose
]
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 8. Problem 7 -- Workflow vs. agent (written)

Both graphs you built are **workflows**: the developer wired every edge, and re-running takes the
same path. In a sentence or two: **what single change would turn one of them into an agent**, and
who would then control the path? (Hint: think about a *back-edge* and who decides whether to take
it. This is exactly the line into L14.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 9. Self-check

Compare against `L1106_lab_solutions.ipynb`. You're done when: `route_app` classifies then routes to
one branch and always reaches `END`; the same ticket takes the same path twice; the user-input graph
routes on `user_choice` with **no model in the decision** (a technical ticket with
`user_choice="billing"` lands in `billing`); the optional eval reports a pass rate with the ambiguous
case failing; and you can name the single change (a model-driven back-edge) that makes a workflow an
agent.

[↑ Back to top](#top)